# Content-Based Music Recommendation

## Objective

Build a scalable content-based music recommendation system using Spotify audio features.

Instead of computing the similarity between every pair of songs, we use the Nearest Neighbors algorithm with cosine distance to efficiently find the most similar songs.

The trained recommendation model will be saved and later deployed using Streamlit.

In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.neighbors import NearestNeighbors

In [2]:
df = pd.read_csv("../data/processed/music_features.csv")

df.head()

,track_id,track_name,artists,album_name,track_genre,popularity,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,5SuOikwiRyPMVoIQDJUgSV,Comedy,Gen Hoshino,Comedy,acoustic,73,0.629244,-0.717148,0.300828,0.551848,-0.850202,-0.504109,0.758743,0.929306,-1.141863
1,4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,Ben Woodward,Ghost (Acoustic),acoustic,55,-0.845908,-1.889980,-1.784744,-0.078993,1.831732,-0.504094,-0.591211,-0.798690,-1.489717
2,1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,Ingrid Michaelson;ZAYN,To Begin Again,acoustic,57,-0.742186,-1.122669,-0.293288,-0.273826,-0.315499,-0.504112,-0.507167,-1.365688,-1.528312
3,6lfxq3CG4xtTiEg7opyCyx,Can't Help Falling In Love,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,acoustic,71,-1.733304,-2.312994,-2.039252,-0.457309,1.774593,-0.503883,-0.428376,-1.276974,1.987859
4,5vjLSffimiIP26QG5WcN2K,Hold On,Chord Overstreet,Hold On,acoustic,82,0.295030,-0.788711,-0.282750,-0.303145,0.463399,-0.504112,-0.686285,-1.184403,-0.073348


In [3]:
df.shape

(114000, 15)

In [4]:
df.columns

Index(['track_id', 'track_name', 'artists', 'album_name', 'track_genre',
       'popularity', 'danceability', 'energy', 'loudness', 'speechiness',
       'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'],
      dtype='str')

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 15 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   track_id          114000 non-null  str    
 1   track_name        114000 non-null  str    
 2   artists           114000 non-null  str    
 3   album_name        114000 non-null  str    
 4   track_genre       114000 non-null  str    
 5   popularity        114000 non-null  int64  
 6   danceability      114000 non-null  float64
 7   energy            114000 non-null  float64
 8   loudness          114000 non-null  float64
 9   speechiness       114000 non-null  float64
 10  acousticness      114000 non-null  float64
 11  instrumentalness  114000 non-null  float64
 12  liveness          114000 non-null  float64
 13  valence           114000 non-null  float64
 14  tempo             114000 non-null  float64
dtypes: float64(9), int64(1), str(5)
memory usage: 22.3 MB


**Separate Song Information and Features**

In [6]:
# Song information:

song_info = df[
    [
        "track_id",
        "track_name",
        "artists"
    ]
]

In [7]:
#ML features: 
feature_columns = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

music_features = df[feature_columns]

music_features.head()

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,0.629244,-0.717148,0.300828,0.551848,-0.850202,-0.504109,0.758743,0.929306,-1.141863
1,-0.845908,-1.889980,-1.784744,-0.078993,1.831732,-0.504094,-0.591211,-0.798690,-1.489717
2,-0.742186,-1.122669,-0.293288,-0.273826,-0.315499,-0.504112,-0.507167,-1.365688,-1.528312
3,-1.733304,-2.312994,-2.039252,-0.457309,1.774593,-0.503883,-0.428376,-1.276974,1.987859
4,0.295030,-0.788711,-0.282750,-0.303145,0.463399,-0.504112,-0.686285,-1.184403,-0.073348


# Build The Recommendation Model

In [8]:
model = NearestNeighbors(
    n_neighbors=11,
    metric="cosine",
    algorithm="brute"
)

model.fit(music_features)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",11
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


**Why 11?**  

Because

* 1 song = itself
* Remaining 10 = recommendations

In [9]:
#save the model
with open("../models/music_recommender.pkl","wb") as file:
    pickle.dump(model,file)

In [10]:
import os

os.listdir("../models")

['music_dataset.csv', 'music_recommender.pkl', 'similarity_matrix.pkl']

## Second Model will recommend song by genre filtering.

In [11]:
from sklearn.neighbors import NearestNeighbors

def recommend_songs(song_name, genre=None, top_n=10):

    # If genre is selected, filter the dataset
    if genre is not None:
        genre_df = df[df["track_genre"] == genre].reset_index(drop=True)

        if genre_df.empty:
            return "No songs found in this genre."
    else:
        # Use the entire dataset
        genre_df = df.copy().reset_index(drop=True)

    # Audio features
    feature_columns = [
        "danceability",
        "energy",
        "loudness",
        "speechiness",
        "acousticness",
        "instrumentalness",
        "liveness",
        "valence",
        "tempo"
    ]

    genre_features = genre_df[feature_columns]

    # Build Nearest Neighbors model only for this genre
    model = NearestNeighbors(
        metric="cosine",
        algorithm="brute"
    )

    model.fit(genre_features)

    # Find selected song
    matches = genre_df[
        genre_df["track_name"].str.lower() == song_name.lower()
    ]

    if matches.empty:
        return f'"{song_name}" not found in {genre} genre.'

    song_index = matches.index[0]

    distances, indices = model.kneighbors(
        genre_features.iloc[[song_index]],
        n_neighbors=min(top_n + 1, len(genre_df))
    )

    recommendations = genre_df.iloc[
        indices.flatten()[1:]
    ][[
        "track_name",
        "artists",
        "album_name",
        "track_genre",
        "popularity"
    ]]

    return recommendations

In [12]:
recommendations  = recommend_songs("Perfect")
recommendations

,track_name,artists,album_name,track_genre,popularity
45158,Perfect,Acoustic Guitar Collective,"Golden Covers, Vol. 2",guitar,39
45907,Ao Clarear,Dieter Huber,Mare,guitar,44
45721,Samba Triste,Baden Powell,Baden Powell À Vontade,guitar,24
45811,Anjuli,Hapa,HAPA,guitar,21
45904,Satellites,Liam Stoler,West End,guitar,46
45817,Estrellita,Gilberto Puente,Historia de Amor,guitar,20
26581,Mind Over Matter,Billboard Baby Lullabies,Lullaby Renditions of Winnie the Pooh,disney,21
45114,Camino Del Sol,Ceriumidis,Camino Del Sol,guitar,30
75748,September Moon,Tony O'Connor,Memento,new-age,21
45901,Sancy,Oli Bloom,Sancy,guitar,45


In [13]:
recommendations = recommend_songs(
    song_name="Spring Day",
    genre="k-pop"
)

recommendations

,track_name,artists,album_name,track_genre,popularity
720,Spring Day,BTS,Proof,k-pop,63
605,INVU,TAEYEON,INVU - The 3rd Album,k-pop,72
396,Answer : Love Myself,BTS,Love Yourself 結 'Answer',k-pop,69
993,Wave,ATEEZ,TREASURE EP.3: One To All,k-pop,63
738,Crystal Snow,BTS,FACE YOURSELF,k-pop,65
310,0X1=LOVESONG (I Know I Love You) feat. Seori,TOMORROW X TOGETHER,The Chaos Chapter: FREEZE,k-pop,74
914,Autumn Leaves,BTS,The Most Beautiful Moment in Life Pt.2,k-pop,65
219,Forever Young,BLACKPINK,SQUARE UP,k-pop,67
421,Lie,BTS,Wings,k-pop,70
235,Yet To Come (Hyundai Ver.),BTS,Yet To Come (Hyundai Ver.),k-pop,75


**Test The model**

In [14]:
df.to_csv("../models/music_dataset.csv",index=False)

In [2]:
import pickle

with open("../models/music_recommender.pkl","rb") as file:

    loaded_model = pickle.load(file)

loaded_model

print("Model saved successfully!")

Model saved successfully!


In [3]:
print(type(loaded_model))

<class 'sklearn.neighbors._unsupervised.NearestNeighbors'>


## Why Nearest Neighbors?

A full cosine similarity matrix requires comparing every song with every other song, which becomes extremely memory-intensive for large datasets.

The Nearest Neighbors algorithm avoids storing all pairwise similarities. Instead, it efficiently finds the closest songs only when a recommendation is requested.

This approach is more scalable and suitable for real-world recommendation systems.

# Conclusion

A scalable content-based music recommendation system was successfully built using the Nearest Neighbors algorithm.

The model uses cosine distance to identify songs with similar audio characteristics.

The trained model was saved using Pickle and is ready for deployment in a Streamlit web application.